[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/07_end_to_end.ipynb)

# 07 · End-to-end synthetic experiment

The pieces in notebooks 00–06 assemble into one callable, `run_validation_experiment`, also exposed as `multilogue-validate`. It uses only synthetic units and `FakeClient` — safe for CI and Colab without keys — and writes tables, a multiverse decomposition, and an evidence pack under a directory you choose (keep it gitignored).

Use this notebook as a smoke test after installing multilogue, then return to the earlier notebooks when you want to change one aspect at a time.

In [ ]:
#@title Setup (run first)
# Free path: synthetic units + FakeClient (no API key).
# Optional: load("polis", ...) downloads public open data into a local cache.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from IPython.display import display, HTML, Markdown
from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


In [ ]:
from pathlib import Path
from multilogue.experiment import run_validation_experiment
from multilogue import viz
import pandas as pd

out = Path("outputs/notebook_e2e")
summary = run_validation_experiment(out, n_docs_per_group=3, units_per_doc=3, k=4, quiet=True)
display(summary)
tables = out / "tables"
for name in ["yield.csv", "group_compare_lenses.csv", "variance_shares.csv"]:
    p = tables / name
    if p.is_file():
        print("—", name)
        display(pd.read_csv(p).head())
shares = pd.read_csv(tables / "variance_shares.csv")
viz.variance_share_bars(shares, path=FIG / "07_variance_shares.png")
print("Evidence pack:", out / "evidence_pack.html")

## Where next?

- Swap `load("synthetic")` for `load("polis", …)` or `load("gdn_cc", …)` in Access when you want public civic text.
- Replace `FakeClient` with an OpenAI or LiteLLM client for real extraction (checkpoints still apply).
- Keep study-specific PDF parsers, prompts, and headline ℓ in your own package; depend on multilogue for the shared method.